# EEG-to-Text Model Comparison Template

**For external researchers who want to compare their model against the V9+QML baseline on ZuCo.**

## What you need before running this notebook

1. Your trained EEG-to-text model evaluated on ZuCo
2. Your TF BLEU-1 and ROUGE-1 scores (minimum)
3. The file `nat_v9_qml_results.json` from the original project (get it from the paper authors or run the product notebook yourself)
4. An NVIDIA API key (free at [build.nvidia.com](https://build.nvidia.com))

## What this notebook does

- Takes your model metrics as input
- Automatically compares them against V5, V8, V9+QML clean, and V9+QML noisy baselines
- Runs 4 domain-specific agents: Scientist, Comparator, Critic, Synthesiser
- V9+QML noisy is the hardware-realistic variant (DepolarizingChannel + PhaseDamping + MC average)
- Produces a structured JSON report you can cite

## You do NOT need to

- Install PyTorch or re-run inference
- Have access to the ZuCo dataset
- Have the original model checkpoints

Your metrics are enough.


In [7]:
# Cell 1 — Install dependencies (run once)
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import nemoguardrails
    print(f'✅ nemoguardrails {nemoguardrails.__version__}')
except ImportError:
    _pip('nemoguardrails>=0.10.0')
    print('✅ nemoguardrails installed')

try:
    import openai
    print(f'✅ openai {openai.__version__}')
except ImportError:
    _pip('openai>=1.0.0')
    print('✅ openai installed')

print('All dependencies ready.')


✅ nemoguardrails 0.21.0
✅ openai 2.30.0
All dependencies ready.


In [9]:
# Cell 2 — Setup paths and API key
import os, sys
from pathlib import Path

# ── Set your NVIDIA API key ───────────────────────────────────────────────────
os.environ['NVIDIA_API_KEY'] = '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'
os.environ.setdefault('OPENAI_API_KEY', os.environ['NVIDIA_API_KEY'])

# ── Locate this notebook's directory (works regardless of where you open it) ──
# The template notebook lives inside eeg_product/, so Path(".") IS eeg_product/
# We add it to sys.path directly — no need to descend further.
NOTEBOOK_DIR    = Path(".").resolve()          # .../project1/eeg_product
EEG_PRODUCT_DIR = NOTEBOOK_DIR                 # already the right folder
PROJECT_DIR     = NOTEBOOK_DIR.parent          # .../project1

if str(EEG_PRODUCT_DIR) not in sys.path:
    sys.path.insert(0, str(EEG_PRODUCT_DIR))

# ── Point to nat_v9_qml_results.json ─────────────────────────────────────────
# Produced by running nat_eeg_agents_v9_product.ipynb (one level up in project1/)
RESULTS_JSON = str(PROJECT_DIR / "nat_v9_qml_results.json")

# ── Verification ─────────────────────────────────────────────────────────────
key = os.environ.get("NVIDIA_API_KEY", "")
print(f"API key set    : {key.startswith('nvapi-') and len(key) > 20}")
print(f"eeg_product at : {EEG_PRODUCT_DIR}")
print(f"Results JSON   : {Path(RESULTS_JSON).exists()}  ({RESULTS_JSON})")

if not Path(RESULTS_JSON).exists():
    print()
    print("  ⚠  nat_v9_qml_results.json not found.")
    print("  Run nat_eeg_agents_v9_product.ipynb (cells 1-19) first,")
    print(f"  or copy the file to: {RESULTS_JSON}")


API key set    : True
eeg_product at : /home/deeptanshu/Desktop/project1/eeg_product
Results JSON   : True  (/home/deeptanshu/Desktop/project1/nat_v9_qml_results.json)


## Step 1 — Fill in your model details

Fill in everything you have. Only `model_name`, `tf_bleu1_pct`, and `tf_rouge1_pct` are required.
The more you fill in, the richer the agent analysis.


In [10]:
# Cell 3 — SAMPLE DATA: EEGConformer + LoRA (a realistic hypothetical submission)
# This is a realistic sample submission representing a published-style model.
# Replace all values with your own model's actual numbers before running.

from eeg_submission_schema import EEGModelSubmission, load_v9_qml_baseline, load_v9_qml_noisy_baseline

my_model = EEGModelSubmission(

    # ── Required ──────────────────────────────────────────────────────────────
    model_name        = "EEGConformer_LoRA_v1",
    architecture_desc = (
        "EEG Conformer encoder (convolutional patch embedding + 6-head self-attention, "
        "4 transformer layers) replacing GRU-based region encoders. "
        "No MoCo pretraining. LoRA rank=8 applied to GPT-2 blocks [9,10,11]. "
        "Single mean-pool across time before LoRA prefix. "
        "No SR adapter. No HTP. No QML component. "
        "Trained on ZuCo sentence-aware split, same normalisation as V8/V9."
    ),

    # Core TF metrics (%)
    tf_bleu1_pct      = 31.02,   # slightly above V8, below V9
    tf_rouge1_pct     = 36.45,

    # ── Recommended ───────────────────────────────────────────────────────────
    tf_bleu4_pct      = 4.51,
    tf_rougeL_pct     = 31.18,
    fg_bleu1_pct      = 14.87,   # free-generation BLEU-1
    bertscore_f1      = 85.71,
    tf_fg_ratio       = 2.09,    # TF/FG ratio (higher = stronger EEG conditioning)

    # Per-condition BLEU-1 (%)
    # NR = Normal Reading | TSR = Timed Silent | SR = Speed Reading
    per_condition_bleu1 = {
        "NR":  31.44,   # best condition (matches literature)
        "TSR": 33.02,   # mid
        "SR":  26.81,   # hardest — still struggles like all models
    },

    # ── Context ───────────────────────────────────────────────────────────────
    dataset           = "ZuCo",
    val_split         = "sentence-aware TEST_SIZE=0.15 seed=42",  # same as V8/V9
    n_val_samples     = 2032,
    dominant_region   = "Left_Temporal",
    eeg_channels      = 24,
    notes             = (
        "Conformer encoder replaces GRU region encoders. "
        "No quantum component. Trained 20 epochs, AdamW lr=3e-4, batch=8. "
        "Parameter count: ~117M (GPT-2 medium backbone, LoRA adds ~0.8M)."
    ),

    # ── Attention weights (optional — include if you have them) ───────────────
    # Normalised cross-region attention from final transformer layer
    # (should sum to ~1.0 across all regions)
    attention_values = {
        "Left_Temporal":     0.224,   # Wernicke's area — dominant
        "Left_Parieto_Occ":  0.187,   # VWFA — visual word form
        "Central_Parietal":  0.168,   # P300 component
        "Right_Temporal":    0.148,
        "Frontal":           0.139,
        "Occipital":         0.134,
    },

    # Per-condition attention breakdown
    attention_per_cond = {
        "NR": {
            "Left_Temporal":     0.218,
            "Left_Parieto_Occ":  0.192,
            "Central_Parietal":  0.171,
            "Right_Temporal":    0.152,
            "Frontal":           0.136,
            "Occipital":         0.131,
        },
        "TSR": {
            "Left_Temporal":     0.221,
            "Left_Parieto_Occ":  0.185,
            "Central_Parietal":  0.169,
            "Right_Temporal":    0.149,
            "Frontal":           0.141,
            "Occipital":         0.135,
        },
        "SR": {
            "Left_Temporal":     0.234,   # elevated in SR — stress response
            "Left_Parieto_Occ":  0.179,
            "Central_Parietal":  0.163,
            "Right_Temporal":    0.144,
            "Frontal":           0.142,
            "Occipital":         0.138,
        },
    },
)

# ── Preview comparison table before running agents ────────────────────────────
v9qml       = load_v9_qml_baseline(RESULTS_JSON)
v9qml_noisy = load_v9_qml_noisy_baseline(RESULTS_JSON)
my_model.print_summary(v9qml)
print(f"  V9+QML noisy : BLEU-1={v9qml_noisy.get('tf_bleu1_pct','?')}")

# ── Validation ────────────────────────────────────────────────────────────────
warnings = my_model.validate()
if warnings:
    print("⚠  Validation warnings:")
    for w in warnings:
        print(f"   - {w}")
else:
    print("✅ Submission valid — ready to run agents")


✅ V9+QML baseline loaded: BLEU-1=30.62%

  Submission: EEGConformer_LoRA_v1
  Model                  BLEU-1   ROUGE-1
  ────────────────────  ────────  ────────
  V5 baseline             29.24%     33.92%
  V8 baseline             30.40%     35.78%
  V9+QML (ours)          30.62%    35.97%
  YOUR MODEL              31.02%     36.45%
    Δ vs V9+QML           +0.40pp    +0.48pp

✅ Submission valid — ready to run agents


## Step 2 — Run the comparison pipeline

This runs 4 agents:
- **Scientist** — analyses your model on its own merits
- **Comparator** — head-to-head table vs V9+QML with per-metric verdicts
- **Critic** — challenges methodology and statistical significance
- **Synthesiser** — plain-language summary + single most important next step

Each call goes through NeMo Guardrails to ensure outputs stay within EEG research scope.


In [11]:
# Cell 4 — Run comparison pipeline
from comparison_pipeline import run_comparison_pipeline

results = await run_comparison_pipeline(
    submission       = my_model,
    results_json_path = RESULTS_JSON,
    verbose          = True,
)


✅ V9+QML baseline loaded: BLEU-1=30.62%

  Submission: EEGConformer_LoRA_v1
  Model                  BLEU-1   ROUGE-1
  ────────────────────  ────────  ────────
  V5 baseline             29.24%     33.92%
  V8 baseline             30.40%     35.78%
  V9+QML (ours)          30.62%    35.97%
  YOUR MODEL              31.02%     36.45%
    Δ vs V9+QML           +0.40pp    +0.48pp

✅ Guardrails loaded (Colang 1.0) from /home/deeptanshu/Desktop/project1/eeg_product/guardrails_config
  EEG Model Comparison Pipeline
  Submitted model : EEGConformer_LoRA_v1
  Reference       : V9+QML (V5/V8 also in context)
  Endpoint        : https://integrate.api.nvidia.com/v1
  Rails           : NeMo Guardrails

[1/4] Scientist agent...
  ✓  latency=5754.1ms  guard=✅
[2/4] Comparator agent...
  ✓  latency=3540.4ms  guard=✅
[3/4] Critic agent...
  ✓  latency=3715.3ms  guard=✅
[4/4] Synthesiser agent...
  ✓  latency=8915.5ms  guard=✅

── Comparison complete ─────────────────────────────────────
  Model compar

In [ ]:
# Cell 5 — Display results
from IPython.display import Markdown, display

display(Markdown('---\n## Scientist Analysis'))
display(Markdown(results['scientist']))

display(Markdown('---\n## Head-to-Head Comparison (Comparator)'))
display(Markdown(results['comparator']))

display(Markdown('---\n## Critic Review'))
display(Markdown(results['critic']))

display(Markdown('---\n## Plain-Language Synthesis'))
display(Markdown(results['synthesiser']))

# Benchmark summary
ps = results['pipeline_summary']
print(f'\n── Pipeline summary ──────────────────────────────')
print(f'  Total latency  : {ps["total_pipeline_ms"]}ms')
print(f'  Guardrail pass : {ps["guardrail_pass_rate_pct"]}%')
print(f'  Agents run     : {ps["n_agents"]}')

---
## Scientist Analysis

**Analysis of the submitted model: EEGConformer_LoRA_v1**

**1. Architecture analysis:**
The submitted model, EEGConformer_LoRA_v1, introduces several changes to the known chain:
- It replaces the GRU-based region encoders with an EEG Conformer encoder, which consists of a convolutional patch embedding layer and a 6-head self-attention mechanism, followed by 4 transformer layers.
- It removes the MoCo pretraining and the SR adapter.
- It applies LoRA (Low-Rank Adaptation) to the GPT-2 blocks with a rank of 8.
- It uses a single mean-pool across time before the LoRA prefix.

**2. TF metrics evaluation:**
The submitted model achieves the following TF metrics:
- TF BLEU-1: 31.02%
- TF ROUGE-1: 36.45%
- TF ROUGE-L: 31.18%
- TF BERTScore: 85.71%

These metrics indicate that the submitted model outperforms the V8 model (BLEU-1: 30.40%, ROUGE-1: 35.78%) and the V9+QML model (BLEU-1: 30.62%, ROUGE-1: 35.97%) in terms of TF metrics.

**3. Per-condition performance:**
The submitted model's per-condition performance is as follows:
- NR: 31.44%
- TSR: 33.02%
- SR: 26.81%

Compared to the V9+QML model, the submitted model performs better in the NR condition (31.44% vs 30.64%) and the TSR condition (33.02% vs 33.69%), but worse in the SR condition (26.81% vs 27.3%).

**4. TF/FG ratio:**
The TF/FG ratio is 2.09, indicating that the EEG conditioning strength is relatively weak compared to the TF metrics.

**5. Missing metrics:**
The following metrics are missing:
- FG BLEU-4
- FG ROUGE-1
- FG ROUGE-L
- BERTScore for V9+QML

These metrics would provide a more comprehensive understanding of the model's performance.

**6. Strengths and weaknesses:**
**Strengths:**
1. The submitted model outperforms the V8 and V9+QML models in terms of TF metrics.
2. It achieves a high BERTScore of 85.71%.
3. It performs well in the NR and TSR conditions.

**Weaknesses:**
1. The TF/FG ratio is relatively low, indicating weak EEG conditioning strength.
2. The submitted model performs worse in the SR condition compared to the V9+QML model.

---
## Head-to-Head Comparison (Comparator)

**Head-to-Head Metric Comparison Table**

| METRIC NAME | Submitted: X% | V9+QML: Y% | Delta: ±Z pp | Verdict: BETTER / WORSE / EQUIVALENT / N/A |
| --- | --- | --- | --- | --- |
| BLEU-1 | 31.02 | 30.62 | ±0.4 | EQUIVALENT |
| BLEU-4 | 4.51 | 4.27 | ±0.24 | EQUIVALENT |
| ROUGE-1 | 36.45 | 35.97 | ±0.48 | EQUIVALENT |
| ROUGE-L | 31.18 | 30.52 | ±0.66 | WORSE |
| FG BLEU-1 | 14.87 | 6.39 | ±8.48 | BETTER |
| BERTScore F1 | 85.71 | null | null | N/A |
| TF FG Ratio | 2.09 | 4.79 | -2.7 | WORSE |
| Per Condition (NR) | 31.44 | 30.64 | ±0.8 | BETTER |
| Per Condition (TSR) | 33.02 | 33.69 | -0.67 | WORSE |
| Per Condition (SR) | 26.81 | 27.3 | -0.49 | WORSE |

**OVERALL VERDICT:** BEATS V9+QML
**STRONGEST IMPROVEMENT:** The submitted model shows a strong improvement in FG BLEU-1, outperforming V9+QML by 8.48%.
**BIGGEST GAP:** The submitted model shows a significant gap in TF FG Ratio, underperforming V9+QML by 2.7.
**FAIRNESS NOTE:** The comparison is considered fair as both models are trained on the same ZuCo sentence-aware split.

---
## Critic Review

**1. Eval protocol**
Problem: The submitted model uses a sentence-aware val split with TEST_SIZE=0.15 and seed=42, but it is unclear if this is the same split used for V9+QML.
Fix: Clarify the val split used for V9+QML to ensure a fair comparison.

**2. Metric completeness**
Problem: The submitted model provides BERTScore F1, but it is missing FG BLEU.
Fix: Include FG BLEU in the evaluation to provide a more comprehensive comparison.

**3. Architecture comparison fairness**
Problem: The submitted model has a significantly larger number of parameters (~117M) compared to V9+QML.
Fix: Consider using a smaller model or a more efficient architecture to ensure a fair comparison.

**4. Per-condition analysis**
Problem: The submitted model performs worse in SR condition compared to V9+QML.
Fix: Investigate the reasons behind the poor performance in SR condition and consider improving the model's ability to handle this condition.

**5. Statistical significance**
Problem: The delta in BLEU-1 is ±0.4, which is not statistically significant at the ZuCo scale (~2032 val samples).
Fix: Consider using a more robust evaluation metric or increasing the number of val samples to achieve statistical significance.

**Correctly identified contributions:**
* Introduction of EEG Conformer encoder
* Use of LoRA for GPT-2 blocks
* Single mean-pool across time before LoRA prefix

**Verdict:** CONDITIONAL ACCEPT
**Confidence: 6/10**
The submission shows some promising improvements, but the lack of clarity on the val split, incomplete metric evaluation, and unfair architecture comparison raise concerns.

---
## Plain-Language Synthesis

The submitted model, EEGConformer_LoRA_v1, aims to improve upon the V9+QML architecture by introducing an EEG Conformer encoder, removing the MoCo pretraining and SR adapter, and applying LoRA to the GPT-2 blocks. This change is an attempt to enhance the model's ability to process EEG signals and generate more accurate text descriptions. The model's architecture is a significant departure from the traditional V9+QML chain, which uses GRU-based region encoders and includes MoCo pretraining.

In a head-to-head comparison with V9+QML, EEGConformer_LoRA_v1 shows mixed results. On the one hand, it performs equally well on BLEU-1, BLEU-4, and ROUGE-1 metrics, indicating that it can generate text descriptions with similar accuracy. However, it falls behind on ROUGE-L, suggesting that it may struggle with longer-range dependencies in the text. On the other hand, EEGConformer_LoRA_v1 significantly outperforms V9+QML on the FG BLEU-1 metric, indicating that it can better capture the nuances of EEG signals. The TF FG Ratio metric, however, shows that EEGConformer_LoRA_v1 is less effective at generating text descriptions for specific conditions.

The critic's concerns highlight several issues with the evaluation protocol and metric completeness. Specifically, the val split used for V9+QML is unclear, and the submitted model is missing FG BLEU in the evaluation. These issues raise concerns about the fairness and comprehensiveness of the comparison. A researcher should be cautious when interpreting the results, as the evaluation protocol may not be entirely fair or complete.

To strengthen the model or the evaluation, the submitting researcher should address the critic's concerns by clarifying the val split used for V9+QML and including FG BLEU in the evaluation. Additionally, the researcher should consider using a more comprehensive evaluation protocol that includes a wider range of metrics and conditions. By doing so, they can provide a more accurate and reliable comparison between EEGConformer_LoRA_v1 and V9+QML.

The single most important next step for EEGConformer_LoRA_v1 is to address the critic's concerns by clarifying the val split used for V9+QML and including FG BLEU in the evaluation to ensure a fair and comprehensive comparison.


── Pipeline summary ──────────────────────────────
  Total latency  : 21925.3ms
  Guardrail pass : 100.0%
  Agents run     : 4


In [13]:
# Cell 6 — Save report
from comparison_pipeline import save_comparison_report

output_file = save_comparison_report(results)
print(f'Report saved to: {output_file}')
print('You can share this JSON file with collaborators or cite it in your paper.')


✅ Comparison report saved → comparison_eegconformer_lora_v1.json
Report saved to: comparison_eegconformer_lora_v1.json
You can share this JSON file with collaborators or cite it in your paper.


---
## Understanding the output

### Comparator verdicts

| Verdict | Meaning |
|---------|--------|
| BETTER | Your model beats V9+QML by > 0.5pp |
| EQUIVALENT | Within ±0.5pp (not meaningfully different) |
| WORSE | Your model is below V9+QML by > 0.5pp |
| N/A | Metric not provided in your submission |

### Per-condition interpretation

The three ZuCo conditions test different cognitive states:
- **NR (Normal Reading)**: Standard reading, easiest for decoding
- **TSR (Timed Silent Read)**: Time-pressured silent reading, moderate
- **SR (Speed Reading)**: Maximum speed, hardest for the decoder

Improvement on SR is the most meaningful — all models struggle there.

### V9+QML clean vs noisy

Both QML variants are provided as reference baselines:
- **V9+QML clean** — noiseless statevector simulation (lightning.qubit). Best val loss: 4.1733
- **V9+QML noisy** — hardware-realistic simulation (DepolarizingChannel p=0.01 + PhaseDamping γ=0.02). Best val loss: 4.1729

The 0.0004 val loss gap shows the architecture is hardware-deployable — real quantum noise at this scale causes negligible degradation. When comparing your model, beating either variant is meaningful. Beating V9+QML noisy means your model outperforms the hardware-ready variant.

### What the TF/FG ratio means

- **< 2×**: EEG prefix strongly conditions the LM
- **2–3×**: Partial conditioning (where most models sit)
- **> 3×**: LM largely ignoring the EEG signal

V8 was 1.97×. V9+QML clean and noisy are reported in the comparison. Where does yours sit?

---

## Questions and contact

If you have questions about the baseline models, the ZuCo evaluation protocol, or the V5→V8→V9→QML architecture chain, refer to:
- The `nat_eeg_agents_v9_product.ipynb` notebook
- The `eeg_submission_schema.py` file for the full schema documentation
  (includes `V9_QML_NOISY_BASELINE` and `load_v9_qml_noisy_baseline()`)
- The `comparison_pipeline.py` file for agent prompt details
- `hybrid_qml_noisy_v9_best.pt` — the noise-trained checkpoint (request from authors)
